---
# FASE 4 | MODELADO (Modeling)
---

## 4.1 Descripción de modelos

### Regresión Logística
Modelo lineal que estima la probabilidad de pertenencia a una clase mediante la funcion sigmoide. Sirve como **modelo de referencia (baseline)** por su simplicidad e interpretabilidad. Requiere que las variables esten escaladas (StandardScaler ya aplicado).

### Random Forest
Ensamble de arboles de decision entrenados con *bootstrap* sobre subconjuntos aleatorios de datos y caracteristicas (*bagging*). La prediccion final es el voto mayoritario. Robusto al ruido, maneja bien relaciones no lineales y provee importancia de caracteristicas.

### Gradient Boosting
Ensamble secuencial donde cada arbol corrige los errores del anterior minimizando una funcion de perdida mediante descenso de gradiente. `learning_rate` bajo (0.05) con `n_estimators` alto (200) equilibra bias-varianza. `subsample=0.8` agrega estocasticidad para mayor regularizacion.

## 4.2 Validacion cruzada estratificada (5-fold)

**StratifiedKFold** garantiza que cada fold tenga la misma proporcion de clases que el conjunto completo. Se reportan media y desviacion estandar de cada metrica para evaluar estabilidad del modelo.

In [ ]:
# Instanciar modelos base
base_models = get_base_models(random_state=RANDOM_STATE)

print('Iniciando búsqueda de hiperparámetros (RandomizedSearchCV)...')
print('Nota: Esto puede tomar unos minutos dependiendo de los recursos del sistema.\n')

# Ejecutar el tuning
best_models = tune_hyperparameters(
    base_models, X_train_proc, y_train.values,
    n_iter=15, cv_folds=3, random_state=RANDOM_STATE
)

# Guardar el reporte de los mejores hiperparámetros en disco
save_tuning_report(best_models, save_path='../reports/best_hyperparameters.json')

In [ ]:
print('\nIniciando validación cruzada estratificada (5-fold) sobre los modelos OPTIMIZADOS...')

cv_results = cross_validate_models(
    best_models, X_train_proc, y_train.values,
    cv_folds=5, random_state=RANDOM_STATE
)

In [ ]:
print('\nResumen de Validacion Cruzada:')
cols_show = ['ROC-AUC (val)', 'ROC-AUC std', 'F1 (val)', 'F1 std',
             'Accuracy (val)', 'Accuracy std', 'Precision (val)', 'Recall (val)',
             'Accuracy (train)', 'Tiempo (s)']
display(cv_results[cols_show].round(4))

In [ ]:
plot_cv_comparison(cv_results, save_path=f'{FIGURES_DIR}/cv_comparison.png')

## 4.3 Analisis de sobreajuste (overfitting)

La diferencia entre `Accuracy (train)` y `Accuracy (val)` en la validacion cruzada indica el grado de sobreajuste.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))

x = np.arange(len(cv_results))
w = 0.35
colors_train = ['#4C72B0', '#55A868', '#C44E52']
colors_val   = ['#9FC8FF', '#AAE8C0', '#FF9999']

for i, (name, row) in enumerate(cv_results.iterrows()):
    ax.bar(x[i] - w/2, row['Accuracy (train)'], w, label=f'{name} (train)' if i==0 else '',
           color=colors_train[i], alpha=0.9)
    ax.bar(x[i] + w/2, row['Accuracy (val)'],   w, label=f'{name} (val)' if i==0 else '',
           color=colors_val[i],   alpha=0.9)

ax.set_xticks(x)
ax.set_xticklabels(cv_results.index, rotation=10)
ax.set_ylabel('Accuracy')
ax.set_ylim(0, 1.1)
ax.set_title('Accuracy Train vs Validacion - Analisis de Sobreajuste', fontweight='bold')
ax.grid(True, axis='y', alpha=0.3)

# Leyenda manual
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor=colors_train[i], label=f'{name} train') for i, name in enumerate(cv_results.index)]
legend_elements += [Patch(facecolor=colors_val[i], label=f'{name} val') for i, name in enumerate(cv_results.index)]
ax.legend(handles=legend_elements, loc='lower right', fontsize=8)

plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}/overfitting_analysis.png', bbox_inches='tight')
plt.show()

print('\nDiferencia Train - Val (indica sobreajuste):')
for name, row in cv_results.iterrows():
    diff = row['Accuracy (train)'] - row['Accuracy (val)']
    print(f'  {name:25}: {diff:+.4f}')

## 4.4 Entrenamiento final sobre todo el conjunto de entrenamiento

In [ ]:
trained_models = {}
for name, model in best_models.items():
    trained_models[name] = train_and_save(
        model, name, X_train_proc, y_train.values,
        models_dir='../models'
    )